# Deep Hedging AI
### Vanilla Deep Hedging engine reference implementation (JAX port)

This is the main example notebook. It shows learning to hedge a vanilla call option (ATM by default), first in a Black & Scholes world with statistical drift, and secondly in a world where a second option can be traded.
The examples are not intended to be overly realistic.

In the Black & Scholes case we see that the hedge learned is _not_ the risk-neutral hedge. The notebook <tt>trainer-bs.ipynb</tt> demonstrates that if the statistical drift is zero, and step size is sufficiently small (daily), then the Deep Hedging hedge approximates the risk-neutral hedge.

Port of `notebooks/trainer.ipynb` by Hans Buehler, June 30 2022.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from deephedging.world import WorldConfig, simulate
from deephedging.model import AgentConfig, GymConfig
from deephedging.train import TrainConfig, fit
from deephedging.plotting import plot_training, plot_blackscholes

# DH_EPOCHS overrides every epoch count for quick smoke runs; DH_CHECKPOINTS="" turns checkpoints off.
epochs = lambda n: int(os.environ.get("DH_EPOCHS") or n)
CHECKPOINTS = os.environ.get("DH_CHECKPOINTS", "checkpoints") or None

## Deep Hedging in a Black & Scholes World

In [ ]:
world_cfg = WorldConfig(samples=10000, steps=20, black_scholes=True)
gym = GymConfig(agent=AgentConfig(depth=3, activation="softplus"), utility="cvar", lmbda=1.)
train_cfg = TrainConfig(epochs=epochs(800), batch_size=32)

world = simulate(world_cfg)
val_world = world.clone(samples=world.nSamples // 10, seed=1)
bs = fit(gym, train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(bs.result, bs.train, bs.val, world, val_world)
plt.show()
print("Keys of the dictionary returned by the rollout: ", list(bs.train))

#### Comparison to Black & Scholes

Note that we use medium risk aversion, hence the hedge is not expected to match the risk-neutral hedge. See <tt>blackscholes.ipynb</tt> for experiments approximating the Black & Scholes solution.

In [ ]:
plot_blackscholes(world, bs.train, gym, vol=world_cfg.rvol)
plt.show()

## Stochastic Vol
### Trading with two assets

In [ ]:
world_cfg = WorldConfig(samples=10000, steps=20, black_scholes=False)
train_cfg = TrainConfig(epochs=epochs(800), batch_size=32, learning_rate=0.001, clipvalue=1., global_clipnorm=1.)

world = simulate(world_cfg)
val_world = world.clone(samples=world.nSamples // 10, seed=1)
sv = fit(gym, train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(sv.result, sv.train, sv.val, world, val_world)
plt.show()